# 4.6 RQ3 — Detection Effectiveness vs Computational Overhead (Colab benchmark)

Chạy **Runtime → Run all**. Notebook sẽ:
1. Lấy `RFID-ExSim-dataset.zip` từ Google Drive (hoặc hỏi để upload nếu không tìm thấy).
2. Dựng lại dữ liệu E1/E2/E3 đúng như notebook `RFID_Evidence_Ablation_Colab_1.ipynb` (S1, S2 = Real; S3 = Clone), với nhóm theo thẻ vật lý.
3. Đo Recall, FPR, thời gian xử lý đặc trưng, thời gian suy luận và cấu trúc mô hình trên 5 seed.
4. In **Bảng 4.y** và **toàn bộ mục 4.6** đã điền số đo và cấu hình máy Colab, đồng thời tải về file `4_6_colab.md`.

Không cần GPU. Thời gian chạy khoảng 3–6 phút.

In [ ]:
# ============================================================
# CELL 1 — Cấu hình: sửa ZIP_PATH nếu file zip nằm chỗ khác trong Drive
# ============================================================
ZIP_PATH = "/content/drive/MyDrive/RFID-ExSim-dataset.zip"
SEEDS = [42, 7, 123, 2024, 99]
REPS = 200

In [ ]:
# ============================================================
# CELL 2 — Lấy dữ liệu
# ============================================================
import os, re, sys, glob, json, time, zipfile, pickle, platform, subprocess
import numpy as np
import pandas as pd
import sklearn

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    if not os.path.exists(ZIP_PATH):
        print("Không tìm thấy", ZIP_PATH, "- hãy chọn file RFID-ExSim-dataset.zip để upload")
        up = files.upload()
        ZIP_PATH = os.path.join("/content", next(iter(up)))
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content/rfid")
    SEARCH_ROOT = "/content/rfid"
else:
    SEARCH_ROOT = os.environ["RFID_DATA_SEARCH"]

hits = glob.glob(os.path.join(SEARCH_ROOT, "**", "data", "processed", "s2_collisions_summary.csv"), recursive=True)
assert hits, "Không tìm thấy data/processed/s2_collisions_summary.csv trong file zip"
ROOT = hits[0].split(os.sep + "data" + os.sep + "processed")[0]
PROC = os.path.join(ROOT, "data", "processed")
print("Dataset root:", ROOT)

In [ ]:
# ============================================================
# CELL 3 — Dựng dữ liệu E1/E2/E3 (logic giống notebook của nhóm)
# ============================================================
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

s1 = pd.read_csv(os.path.join(PROC, "meta_sessions_S1.csv"))
s2_coll = pd.read_csv(os.path.join(PROC, "s2_collisions_summary.csv"))
s3 = pd.read_csv(os.path.join(PROC, "s3_clone_collisions_summary.csv"))
plan = pd.read_csv(os.path.join(ROOT, "data", "plans", "s3_plan.csv"))


def protocol_features_from_collision_events(df, session_col, dt_col):
    g = df.groupby(session_col)[dt_col]
    out = g.agg(collision_count="count", dt_mean="mean", dt_std="std", dt_median="median",
                dt_min="min", dt_max="max",
                dt_abs_mean=lambda x: np.abs(x).mean(),
                dt_abs_median=lambda x: np.abs(x).median()).reset_index()
    by = df[[session_col, dt_col]].groupby(session_col)[dt_col]
    extra = pd.DataFrame({
        session_col: by.apply(lambda x: 0).index,
        "small_dt_10ms_ratio": by.apply(lambda x: (np.abs(x) <= 0.010).mean()).values,
        "small_dt_50ms_ratio": by.apply(lambda x: (np.abs(x) <= 0.050).mean()).values,
        "positive_dt_ratio": by.apply(lambda x: (x > 0).mean()).values,
        "negative_dt_ratio": by.apply(lambda x: (x < 0).mean()).values,
        "zero_dt_ratio": by.apply(lambda x: (x == 0).mean()).values,
    })
    return out.merge(extra, on=session_col, how="left")


PROTOCOL_FEATURES = ["collision_count", "dt_mean", "dt_std", "dt_median", "dt_min", "dt_max",
                     "dt_abs_mean", "dt_abs_median", "small_dt_10ms_ratio", "small_dt_50ms_ratio",
                     "positive_dt_ratio", "negative_dt_ratio", "zero_dt_ratio"]
E1 = PROTOCOL_FEATURES
E2 = ["read_count"]
E3 = PROTOCOL_FEATURES + ["read_count"]

s2_protocol = protocol_features_from_collision_events(s2_coll, "session_id", "dt"); s2_protocol["label"] = 0
s3w = s3.copy(); s3w["dt_sec"] = s3w["delta_ms"] / 1000.0
s3_protocol = protocol_features_from_collision_events(s3w, "session_id", "dt_sec"); s3_protocol["label"] = 1
real_b = s1.assign(read_count=s1["total_reads"], label=0)[["session_id", "read_count", "label"]]
clone_b = s3.groupby("session_id").size().rename("read_count").reset_index().assign(label=1)

protocol_data = pd.concat([s2_protocol[["session_id"] + E1 + ["label"]],
                           s3_protocol[["session_id"] + E1 + ["label"]]], ignore_index=True)
behavior_data = pd.concat([real_b, clone_b], ignore_index=True)


def extract_tag(x):
    m = re.search(r"(TAG\d+)", str(x))
    return m.group(1) if m else str(x)


def make_sample_index(df):
    out = df.copy()
    out["tag_group"] = out["session_id"].map(extract_tag)
    out["sample_key"] = out["tag_group"] + "__" + out.groupby("tag_group").cumcount().astype(str)
    return out


protocol_data, behavior_data = make_sample_index(protocol_data), make_sample_index(behavior_data)
keys = set(protocol_data.sample_key) & set(behavior_data.sample_key)
ed = protocol_data[protocol_data.sample_key.isin(keys)][["sample_key", "session_id", "label"] + E1].merge(
    behavior_data[behavior_data.sample_key.isin(keys)][["sample_key", "read_count"]], on="sample_key")

# Nhóm theo thẻ vật lý: phiên S3 (…_P10_run1) được ánh xạ về TAG qua s3_plan.csv
p2t = dict(zip(plan.pair_id, plan.tag_id))
ed["tag"] = [extract_tag(s) if "TAG" in s else p2t[re.search(r"_(P\d+)_", s).group(1)] for s in ed.session_id]
print("Mẫu:", ed.shape[0], "| Real/Clone:", ed.label.value_counts().to_dict(), "| nhóm thẻ:", ed.tag.nunique())


def group_split(df, cols, seed, test_size=0.2, val_size=0.2):
    X, y, g = df[cols], df["label"], df["tag"]
    tv, te = next(GroupShuffleSplit(1, test_size=test_size, random_state=seed).split(X, y, g))
    Xtv, ytv, gtv = X.iloc[tv].reset_index(drop=True), y.iloc[tv].reset_index(drop=True), g.iloc[tv].reset_index(drop=True)
    tr, _ = next(GroupShuffleSplit(1, test_size=val_size / (1 - test_size), random_state=seed + 1).split(Xtv, ytv, gtv))
    return (Xtv.iloc[tr].reset_index(drop=True), ytv.iloc[tr].reset_index(drop=True),
            X.iloc[te].reset_index(drop=True), y.iloc[te].reset_index(drop=True))


def make_rf(seed):
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("rf", RandomForestClassifier(n_estimators=300, random_state=seed, class_weight="balanced"))])


def make_svm():
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()),
                     ("svm", SVC(kernel="rbf", class_weight="balanced", random_state=42))])


def fit_protocol_detector(X, y):
    scores = {}
    for c in X.columns:
        a, b = X.loc[y == 0, c].dropna(), X.loc[y == 1, c].dropna()
        pooled = np.sqrt((a.var() + b.var()) / 2)
        scores[c] = 0.0 if (pooled == 0 or np.isnan(pooled)) else abs((a.mean() - b.mean()) / pooled)
    f = max(scores, key=scores.get)
    a, b = X.loc[y == 0, f].median(), X.loc[y == 1, f].median()
    return {"feature": f, "threshold": (a + b) / 2, "real_is_low": a < b}


def predict_protocol(X, d):
    x = X[d["feature"]].fillna(d["threshold"])
    return ((x > d["threshold"]) if d["real_is_low"] else (x < d["threshold"])).astype(int).to_numpy()

In [ ]:
# ============================================================
# CELL 4 — Đo hiệu quả và chi phí (khoảng 3–5 phút)
# ============================================================
def median_ms(fn, reps=REPS):
    fn()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts) * 1000)


def rates(y, p):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return tp / (tp + fn), fp / (fp + tn)


rows, tree_nodes = [], []
for s in SEEDS:
    Xtr, ytr, Xte, yte = group_split(ed, E1, s)
    det = fit_protocol_detector(Xtr, ytr)
    r, f = rates(yte, predict_protocol(Xte, det))
    one = Xte.iloc[[0]]
    rows.append(dict(method="E1", seed=s, recall=r, fpr=f, infer=median_ms(lambda: predict_protocol(one, det)),
                     batch=np.nan, feature=det["feature"]))
    for cfg, cols in [("E2", E2), ("E3", E3)]:
        Xtr, ytr, Xte, yte = group_split(ed, cols, s)
        for mname, mk in [("RF", lambda: make_rf(s)), ("SVM", make_svm)]:
            m = mk().fit(Xtr, ytr)
            r, f = rates(yte, m.predict(Xte))
            one = Xte.iloc[[0]]
            row = dict(method=f"{cfg}-{mname}", seed=s, recall=r, fpr=f,
                       infer=median_ms(lambda: m.predict(one)),
                       batch=median_ms(lambda: m.predict(Xte), reps=50) / len(Xte))
            if mname == "RF":
                est = m.named_steps["rf"].estimators_
                row["nodes"] = sum(t.tree_.node_count for t in est)
                tree_nodes += [t.tree_.node_count for t in est]
            else:
                row["n_sv"] = int(m.named_steps["svm"].n_support_.sum())
            rows.append(row)
    print("xong seed", s)
R = pd.DataFrame(rows)

# Thời gian xử lý đặc trưng cho một phiên (gọi đơn lẻ) và theo lô
s2_ids, s3_ids = s2_coll.session_id.unique()[:40], s3w.session_id.unique()[:40]
prot = [median_ms(lambda g=s2_coll[s2_coll.session_id == i]: protocol_features_from_collision_events(g, "session_id", "dt"), 30) for i in s2_ids]
prot += [median_ms(lambda g=s3w[s3w.session_id == i]: protocol_features_from_collision_events(g, "session_id", "dt_sec"), 30) for i in s3_ids]
proc_protocol = float(np.median(prot))
proc_behavior = float(np.median([median_ms(lambda g=s3[s3.session_id == i]: len(g), 200) for i in s3_ids]))
t0 = time.perf_counter(); protocol_features_from_collision_events(s2_coll, "session_id", "dt")
proc_protocol_batch = (time.perf_counter() - t0) * 1000 / s2_coll.session_id.nunique()


def load_raw(s):
    df = pd.DataFrame([json.loads(l) for l in open(os.path.join(PROC, f"rfid_dataset_{s}_all.jsonl")) if l.strip()])
    df = df[df.event == "read_success"].copy()
    ts = pd.to_datetime(df.timestamp_iso, utc=True)
    df["t"] = (ts - pd.Timestamp("2025-11-01", tz="UTC")).dt.total_seconds()
    return df


def pair_cross_reader(g, w=0.005):
    t = np.round(g["t"].values * 1000) / 1000
    a = np.sort(t[g.device_id.values == "ESP32_A"]); b = np.sort(t[g.device_id.values == "ESP32_B"])
    i = np.clip(np.searchsorted(b, a), 1, len(b) - 1)
    nn = np.where(np.abs(b[i] - a) < np.abs(b[i - 1] - a), b[i], b[i - 1])
    d = nn - a
    return d[np.abs(d) <= w + 1e-9]


raw = {s: load_raw(s) for s in ["S1", "S2", "S3"]}
dur = {s: float((df.groupby("session_id").t.max() - df.groupby("session_id").t.min()).median()) for s, df in raw.items()}
rr = pd.concat([raw["S2"], raw["S3"]])
proc_pairing = float(np.median([median_ms(lambda g=rr[rr.session_id == i]: pair_cross_reader(g), 30)
                                for i in rr.session_id.unique()[::6]]))
print("Hoàn tất đo.")

In [ ]:
# ============================================================
# CELL 5 — Cấu hình máy Colab
# ============================================================
def cpu_model():
    try:
        out = subprocess.run(["lscpu"], capture_output=True, text=True).stdout
        m = re.search(r"Model name:\s*(.+)", out)
        if m:
            return re.sub(r"\s+", " ", m.group(1)).strip()
    except Exception:
        pass
    return platform.processor() or platform.machine()


ENV = dict(cpu=cpu_model(), vcpu=os.cpu_count(), py=platform.python_version(),
           sk=".".join(sklearn.__version__.split(".")[:2]))
ENV_LABEL = "Google Colab" if IN_COLAB else "máy cục bộ"
print(ENV_LABEL, ENV)

In [ ]:
# ============================================================
# CELL 6 — Bảng 4.y và mục 4.6 hoàn chỉnh (tải về 4_6_colab.md)
# ============================================================
import math

A = R.groupby("method").agg(recall=("recall", "mean"), recall_sd=("recall", "std"),
                            fpr=("fpr", "mean"), fpr_sd=("fpr", "std"),
                            infer=("infer", "median"), batch=("batch", "median"),
                            nodes=("nodes", "median") if "nodes" in R else ("seed", "size"),
                            n_sv=("n_sv", "median") if "n_sv" in R else ("seed", "size"))
f2 = lambda x: f"{x:.2f}"
f3 = lambda x: f"{x:.3f}"
proc_b = "< 0.001" if proc_behavior < 0.001 else f"{proc_behavior:.3f}"
proc_p = f2(proc_protocol)

table_rows = [
    ("MB/MCM (E1, ngưỡng giao thức)", "E1", proc_p, "1 đặc trưng, 1 ngưỡng"),
    ("RF (E2)", "E2-RF", proc_b, f"300 cây, {int(A.loc['E2-RF','nodes'])} nút"),
    ("SVM (E2)", "E2-SVM", proc_b, f"{int(A.loc['E2-SVM','n_sv'])} vectơ hỗ trợ"),
    ("Hybrid – RF (E3)", "E3-RF", proc_p, f"300 cây, {int(A.loc['E3-RF','nodes'])} nút"),
    ("Hybrid – SVM (E3)", "E3-SVM", proc_p, f"{int(A.loc['E3-SVM','n_sv'])} vectơ hỗ trợ"),
]
lines = ["| Phương pháp | Recall | FPR | Thời gian xử lý (ms/phiên) | Thời gian suy luận (ms/quyết định) | Cấu trúc mô hình |",
         "|---|---|---|---|---|---|"]
for name, k, proc, struct in table_rows:
    inf = f2(A.loc[k, "infer"]) + ("" if k == "E1" else f" ({f2(A.loc[k, 'batch'])})")
    lines.append(f"| {name} | {f3(A.loc[k,'recall'])} | {f3(A.loc[k,'fpr'])} | {proc} | {inf} | {struct} |")
TABLE = "\n".join(lines)

perfect = bool((R.recall == 1).all() and (R.fpr == 0).all())
if not perfect:
    print("CẢNH BÁO: Recall/FPR không còn tuyệt đối — đoạn thảo luận thứ hai cần viết lại theo số mới.")

hyb_rf = proc_protocol + A.loc["E3-RF", "infer"]
hyb_svm = proc_protocol + A.loc["E3-SVM", "infer"]
svm_lo = min(A.loc["E2-SVM", "infer"], A.loc["E3-SVM", "infer"])
svm_hi = max(A.loc["E2-SVM", "infer"], A.loc["E3-SVM", "infer"])
rf_inf = (A.loc["E2-RF", "infer"] + A.loc["E3-RF", "infer"]) / 2
sv_lo, sv_hi = int(min(A.loc["E2-SVM", "n_sv"], A.loc["E3-SVM", "n_sv"])), int(max(A.loc["E2-SVM", "n_sv"], A.loc["E3-SVM", "n_sv"]))
max_nodes = int(max(A.loc["E2-RF", "nodes"], A.loc["E3-RF", "nodes"]))
per_tree = sorted(set(tree_nodes))
tree_sentence = ("mỗi cây trong rừng chỉ có ba nút, tức một phép so sánh ngưỡng duy nhất, nên 300 cây là dư thừa"
                 if per_tree == [3] else f"các cây trong rừng chỉ có {per_tree[0]}–{per_tree[-1]} nút")
VN = {1: "một", 2: "hai", 3: "ba", 4: "bốn", 5: "năm", 6: "sáu", 7: "bảy", 8: "tám"}
mag_lo = int(math.floor(math.log10(min(dur.values()) * 1000 / max(hyb_rf, hyb_svm))))
mag_hi = int(math.floor(math.log10(max(dur.values()) * 1000 / A.loc["E1", "infer"])))
behav_txt = "dưới 1 µs" if proc_behavior < 0.001 else f"{proc_behavior:.3f} ms"
env_txt = f"trên {ENV_LABEL} ({ENV['vcpu']} vCPU {ENV['cpu']}, Python {ENV['py']} và scikit-learn {ENV['sk']}) [7]"
sd_txt = ("Recall và FPR có độ lệch chuẩn bằng 0 trên năm hạt giống" if perfect
          else f"Recall và FPR là trung bình trên năm hạt giống")

SECTION = f'''## 4.6. RQ3 – Detection Effectiveness vs Computational Overhead

RQ3 xem xét quan hệ giữa hiệu quả phát hiện, tỷ lệ báo động giả và chi phí tính toán của các bộ phát hiện. Bảng 4.y so sánh bộ phát hiện giao thức E1, đại diện cho baseline MB/MCM, với RF và SVM trên đặc trưng hành vi (E2) và với cấu hình lai (Hybrid, E3). Recall và FPR được tính trên tập kiểm tra và lấy trung bình qua năm hạt giống. Thời gian xử lý là thời gian dựng vectơ đặc trưng cho một phiên từ các bản ghi sự kiện, dùng đúng mã trích xuất đặc trưng của pipeline. Thời gian suy luận là thời gian đưa ra một quyết định cho một vectơ đặc trưng. Cả hai được đo bằng trung vị của {REPS} lần gọi đơn lẻ, {env_txt}.

Bảng 4.y. Hiệu quả phát hiện và chi phí tính toán theo phiên

{TABLE}

{sd_txt}; các mẫu được gom nhóm theo thẻ vật lý. Giá trị trong ngoặc là thời gian trung bình mỗi mẫu khi dự đoán theo lô trên tập kiểm tra.

Do mọi phương pháp đều đạt Recall 1.000 và FPR 0.000, trục hiệu quả và trục báo động giả không phân biệt được các phương pháp, và sự đánh đổi của RQ3 chỉ còn thể hiện trên trục chi phí. Chi phí của bằng chứng giao thức nằm chủ yếu ở khâu trích xuất đặc trưng: tính 13 thống kê va chạm cho một phiên mất {f2(proc_protocol)} ms khi gọi đơn lẻ ({f2(proc_protocol_batch)} ms/phiên khi xử lý theo lô), chưa kể {f2(proc_pairing)} ms/phiên để ghép cặp các lượt đọc chéo từ sự kiện thô. Đặc trưng hành vi `read_count` chỉ là một phép đếm, tốn {behav_txt}. Ở khâu suy luận, ngưỡng E1 mất {f2(A.loc['E1','infer'])} ms, SVM mất {f2(svm_lo)}–{f2(svm_hi)} ms, còn RF mất khoảng {rf_inf:.1f} ms. Khoảng cách của RF chủ yếu do chi phí gọi tuần tự 300 cây trong scikit-learn, vì khi dự đoán theo lô thời gian mỗi mẫu chỉ còn {f2(A.loc['E2-RF','batch'])} ms. Cấu hình Hybrid cộng dồn chi phí của cả hai khâu, tổng cộng khoảng {hyb_rf:.1f} ms/phiên với RF và {hyb_svm:.1f} ms/phiên với SVM, nhưng không cải thiện Recall hay FPR so với E2. Cấu trúc mô hình cũng cho thấy bài toán không đòi hỏi độ phức tạp này: {tree_sentence}, và SVM chỉ cần {sv_lo}–{sv_hi} vectơ hỗ trợ. Kết quả này nhất quán với nhận định ở mục 4.5 rằng hai lớp tách được bằng một ngưỡng trên nhiều đặc trưng khác nhau.

Hai giới hạn quyết định cách diễn giải Bảng 4.y. Thứ nhất, các giá trị Recall và FPR không thể xem là hiệu quả an ninh: mục 4.5 cho thấy đặc trưng mà E1 sử dụng (`negative_dt_ratio`) chỉ còn AUC 0.546 khi hai lớp được xử lý đồng nhất. Vì vậy, Bảng 4.y không cho phép kết luận rằng baseline giao thức đạt cùng mức an ninh với chi phí thấp nhất; phần có giá trị của bảng là các số đo chi phí, vốn không phụ thuộc vào tính hợp lệ của nhãn. Thứ hai, độ trễ ra quyết định trong triển khai bị chi phối bởi thời gian quan sát hơn là thời gian tính toán. Các bộ phát hiện hoạt động ở mức phiên, nên chỉ đưa ra quyết định sau khi phiên kết thúc. Trung vị độ dài phiên là {dur['S1']:.0f} giây ở S1, {dur['S2']:.0f} giây ở S2 và {dur['S3']:.0f} giây ở S3, cao hơn chi phí tính toán từ khoảng {VN.get(mag_lo, mag_lo)} đến {VN.get(mag_hi, mag_hi)} bậc độ lớn. Hơn nữa, các phép đo được thực hiện trên CPU của {ENV_LABEL}, không phải trên ESP32 hay phần cứng tại điểm kiểm soát. Kích thước của các mô hình (tối đa {max_nodes} nút cây hoặc {sv_hi} vectơ hỗ trợ) là nhỏ so với bộ nhớ SRAM 520 KB của ESP32 [8], nhưng chưa có phép đo nào trên thiết bị nhúng. Do đó, kết quả chỉ hỗ trợ nhận định về tính khả thi tính toán cho triển khai định hướng điểm kiểm soát (computational feasibility for checkpoint-oriented deployment), không hỗ trợ tuyên bố về khả năng phát hiện thời gian thực.

### References (tiếp nối mục 4.5)

[7] F. Pedregosa, G. Varoquaux, A. Gramfort, V. Michel, B. Thirion, O. Grisel, M. Blondel, P. Prettenhofer, R. Weiss, V. Dubourg, J. Vanderplas, A. Passos, D. Cournapeau, M. Brucher, M. Perrot, and É. Duchesnay, "Scikit-learn: Machine learning in Python," *Journal of Machine Learning Research*, vol. 12, pp. 2825–2830, 2011.

[8] Espressif Systems, *ESP32 Series Datasheet*, Espressif Systems, Shanghai, China.
'''

print("E1 chọn đặc trưng:", sorted(set(R[R.method == "E1"].feature)))
out_path = "/content/4_6_colab.md" if IN_COLAB else "4_6_colab.md"
with open(out_path, "w", encoding="utf-8") as fh:
    fh.write(SECTION)
try:
    from IPython.display import display, Markdown
    display(Markdown(SECTION))
except Exception:
    print(SECTION)
if IN_COLAB:
    files.download(out_path)